<a href="https://colab.research.google.com/github/usman-stack-322/flyrank-ml-internship-v2/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/usman-stack-322/flyrank-ml-internship-v2/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1 ## 1. Two paper findings + my methodology questions

### Finding 1 — The Anatomy of Growing Content

The paper reports that growing pages were, on average, longer and younger than declining pages: 3.2K vs 2.3K words and 184 vs 230 days old. The label comes from the observed 30-day impression trend: pages with more than 10% growth were classified as “up”, while pages with more than 10% decline were classified as “down”.

**Methodology question:** The validation design supports an observational comparison, but it does not establish that increasing word count or refreshing a page causes growth. The groups may differ in other ways, such as topic, demand, or existing visibility. I would treat this finding as a directional signal for prioritization rather than a causal rule.

### Finding 2 — The Content Performance Curve

The paper reports that content health was highest around 61–90 days and declined substantially around 271–365 days. The label here comes from content-age buckets, with health score compared across different age groups. The 365+ recovery is interpreted cautiously because refreshed older pages may be different from pages that remained unchanged.

**Methodology question:** The age-bucket comparison shows an observed relationship between content age and health, but it does not prove that age itself causes performance to decline. A stronger validation design would control for factors such as content type, topic, and prior visibility, or use a time-aware/grouped validation approach when building a predictive model.




In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Simple checks for the two paper findings

print("Finding 1:")
print("Growing pages were reported as younger and longer than declining pages.")
print("Methodology question: this is observational, so it does not prove causation.")

print("\nFinding 2:")
print("Content health was reported to vary across content-age buckets.")
print("Methodology question: age association does not prove that age causes decline.")


Finding 1:
Growing pages were reported as younger and longer than declining pages.
Methodology question: this is observational, so it does not prove causation.

Finding 2:
Content health was reported to vary across content-age buckets.
Methodology question: age association does not prove that age causes decline.


##  2. My model under an honest split (before/after)

My Week-5 model was first evaluated using the original split. I then re-evaluated the same model using a grouped or time-aware split to reduce the risk of having very similar observations in both training and test data.

The purpose of this comparison is to see whether the model performance remains similar when evaluated under a more realistic validation design.

I compare the **same metric on the same target** before and after the honest split. If performance decreases under the honest split, I treat that gap as useful evidence about how much the original evaluation may have benefited from an easier split.

The honest result is used for decision-support and prioritization, not as proof that the model will perform identically on future data.


In [9]:
import os
import sys
import subprocess
import pandas as pd

REPO_URL = "https://github.com/usman-stack-322/flyrank-ml-internship-v2"
REPO_DIR = "flyrank-ml-internship-v2"

# Clone repo if it is not already available
if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )

# Move into the repo
os.chdir(REPO_DIR)

# Load dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create target
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Current directory:", os.getcwd())
print("Dataset shape:", df.shape)
print("Target created:", "is_declining_label" in df.columns)

Current directory: /content/flyrank-ml-internship-v2/flyrank-ml-internship-v2
Dataset shape: (30000, 45)
Target created: True


In [10]:
# Section 2: Honest time-aware validation

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

# Target
target = "is_declining_label"

# Same features used in Week-5
feature_columns = [
    "search_volume",
    "competition",
    "competition_level",
    "cpc",
    "content_type",
    "main_intent",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "age_tier",
    "age_tier_order",
    "days_since_last_update",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "impression_tier",
    "position_tier",
]

# 80/20 time-aware split
split_point = int(len(df) * 0.80)

train_df = df.iloc[:split_point].copy()
test_df = df.iloc[split_point:].copy()

X_train = train_df[feature_columns]
y_train = train_df[target]

X_test = test_df[feature_columns]
y_test = test_df[target]

# Separate numeric and categorical features
numeric_features = X_train.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

categorical_features = [
    col for col in feature_columns
    if col not in numeric_features
]

# Preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# Same Random Forest settings from Week-5
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

# Train on earlier data
pipeline.fit(X_train, y_train)

# Predict later data
honest_pred = pipeline.predict(X_test)

# Calculate F1
honest_f1 = f1_score(
    y_test,
    honest_pred,
    zero_division=0
)

print("HONEST TIME-AWARE VALIDATION")
print("Training rows:", len(train_df))
print("Test rows:", len(test_df))
print("Honest F1:", round(honest_f1, 3))

HONEST TIME-AWARE VALIDATION
Training rows: 24000
Test rows: 6000
Honest F1: 0.837


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Section 3: Leakage audit

# Features used by the final model
print("FINAL MODEL FEATURES")
print("-" * 40)

for feature in feature_columns:
    print(feature)

# Check for obvious leakage columns
possible_leakage = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "content_id",
    "client_id"
]

print("\nLEAKAGE CHECK")
print("-" * 40)

for column in possible_leakage:
    if column in feature_columns:
        print("WARNING - possible leakage:", column)
    else:
        print("OK - not used as feature:", column)

print("\nLeakage audit complete.")


FINAL MODEL FEATURES
----------------------------------------
search_volume
competition
competition_level
cpc
content_type
main_intent
word_count
char_count
impressions_90d
clicks_90d
pageviews_90d
sessions_90d
users_90d
engaged_sessions_90d
ai_sessions_90d
scroll_events_90d
days_with_impressions
days_with_sessions
impressions_last_30d
clicks_last_30d
sessions_last_30d
impressions_prev_30d
clicks_prev_30d
sessions_prev_30d
content_age_days
age_tier
age_tier_order
days_since_last_update
freshness_tier
word_count_tier
char_count_tier
ctr
avg_position
engagement_rate
scroll_rate
ai_traffic_pct
impression_tier
position_tier

LEAKAGE CHECK
----------------------------------------
OK - not used as feature: trend_direction
OK - not used as feature: trend_pct
OK - not used as feature: is_declining_label
OK - not used as feature: content_id
OK - not used as feature: client_id

Leakage audit complete.


## 4. Claim rewrite
The Random Forest measured an F1 score of 0.837 on the time-aware test split, compared with 0.320 for the Week-4 baseline. This observed result suggests that the model may provide a stronger directional signal for identifying declining pages in this dataset. It should be used for decision-support and prioritization, with human review, rather than as proof that refreshing a page will cause improvement.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Section 4: Claim rewrite check

claim = """
The Random Forest measured an F1 score of 0.837 on the time-aware
test split, compared with 0.320 for the Week-4 baseline.
This observed result suggests that the model may provide a stronger
directional signal for identifying declining pages in this dataset.
It should be used for decision-support and prioritization, with
human review, rather than as proof that refreshing a page will cause improvement.
"""

print(claim)



The Random Forest measured an F1 score of 0.837 on the time-aware
test split, compared with 0.320 for the Week-4 baseline.
This observed result suggests that the model may provide a stronger
directional signal for identifying declining pages in this dataset.
It should be used for decision-support and prioritization, with
human review, rather than as proof that refreshing a page will cause improvement.



## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.